# ARC Prize 2026 — ARC-AGI-3 Submission

Built from `agent/my_agent.py` via `scripts/build_notebook.py`. Do not edit cells directly — edit the source file and re-run `make submit`.

In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatib

In [2]:
%%writefile /tmp/my_agent.py
"""Causal Dice agent for the ARC Prize 2026 ARC-AGI-3 competition.

The policy is game-agnostic. It treats actions as interventions, learns a
compact causal transition model online, and uses deterministic loaded dice to
choose among experiments. Dice weight is expected information gain plus
learned progress value, minus death, repetition, cycle, and action costs.
"""
from __future__ import annotations

import hashlib
import math
import random
from collections import Counter, defaultdict, deque
from typing import Any, Optional

import numpy as np
from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent


class Evidence:
    __slots__ = (
        "attempts", "changes", "no_changes", "deaths", "progress",
        "delta_sum", "novelty_sum", "value",
    )

    def __init__(self) -> None:
        self.attempts = 0
        self.changes = 0
        self.no_changes = 0
        self.deaths = 0
        self.progress = 0
        self.delta_sum = 0.0
        self.novelty_sum = 0.0
        self.value = 0.0


class Candidate:
    __slots__ = (
        "action_id", "key", "model_key", "question", "target",
        "target_label", "target_prior", "utility", "information", "risk",
        "confidence",
    )

    def __init__(
        self,
        action_id: int,
        key: tuple[Any, ...],
        model_key: tuple[Any, ...],
        question: str,
        target: Optional[tuple[int, int]] = None,
        target_label: str = "",
        target_prior: float = 0.0,
    ) -> None:
        self.action_id = action_id
        self.key = key
        self.model_key = model_key
        self.question = question
        self.target = target
        self.target_label = target_label
        self.target_prior = target_prior
        self.utility = 0.0
        self.information = 0.0
        self.risk = 0.0
        self.confidence = 0.0


class Pending:
    __slots__ = ("state_signature", "grid", "levels_completed", "candidate")

    def __init__(
        self,
        state_signature: str,
        grid: np.ndarray,
        levels_completed: int,
        candidate: Candidate,
    ) -> None:
        self.state_signature = state_signature
        self.grid = grid
        self.levels_completed = levels_completed
        self.candidate = candidate


class TraceEntry:
    __slots__ = ("state_signature", "candidate", "changed", "reward")

    def __init__(
        self,
        state_signature: str,
        candidate: Candidate,
        changed: bool,
        reward: float,
    ) -> None:
        self.state_signature = state_signature
        self.candidate = candidate
        self.changed = changed
        self.reward = reward


class MyAgent(Agent):
    """Online causal explorer with deterministic, uncertainty-aware dice."""

    MAX_ACTIONS = 400
    MAX_COMPONENTS = 96
    TRACE_CREDIT_DEPTH = 36

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        digest = hashlib.sha256(self.game_id.encode("utf-8")).digest()
        self.seed = int.from_bytes(digest[:8], "big", signed=False)
        self.rng = random.Random(self.seed)

        self.step_index = 0
        self.current_level = 0
        self.level_start_step = 0
        self.pending: Optional[Pending] = None
        self.recent_change_mask: Optional[np.ndarray] = None
        self.last_effective_action_id: Optional[int] = None
        self.controlled_center: Optional[tuple[int, int]] = None
        self.walkable_colors: Counter[int] = Counter()
        self.navigation_bonus: dict[int, float] = {}

        self.evidence: defaultdict[tuple[Any, ...], Evidence] = defaultdict(Evidence)
        self.state_q: defaultdict[tuple[str, tuple[Any, ...]], float] = defaultdict(float)
        self.role_q: defaultdict[tuple[Any, ...], float] = defaultdict(float)
        self.state_visits: Counter[str] = Counter()
        self.state_action_visits: Counter[tuple[str, tuple[Any, ...]]] = Counter()
        self.transitions: defaultdict[
            tuple[str, tuple[Any, ...]], Counter[str]
        ] = defaultdict(Counter)
        self.trace: deque[TraceEntry] = deque(maxlen=256)

        self.success_macro: list[tuple[int, tuple[Any, ...]]] = []
        self.macro_cursor = 0
        self.progress_events = 0
        self.death_events = 0
        self._trace_enabled = True

    @property
    def name(self) -> str:
        return f"{super().name}.causal-dice-v1.{self.MAX_ACTIONS}"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    @staticmethod
    def _grid_from(frame: FrameData) -> np.ndarray:
        if not frame.frame:
            return np.zeros((1, 1), dtype=np.uint8)
        raw = np.asarray(frame.frame[-1])
        if raw.ndim != 2 or raw.size == 0:
            return np.zeros((1, 1), dtype=np.uint8)
        return np.clip(raw, 0, 255).astype(np.uint8, copy=False)

    @staticmethod
    def _signature(grid: np.ndarray) -> str:
        h = hashlib.blake2b(digest_size=12)
        h.update(int(grid.shape[0]).to_bytes(2, "big"))
        h.update(int(grid.shape[1]).to_bytes(2, "big"))
        h.update(grid.tobytes(order="C"))
        return h.hexdigest()

    @staticmethod
    def _levels(frame: FrameData) -> int:
        try:
            return int(frame.levels_completed)
        except (TypeError, ValueError):
            return 0

    @staticmethod
    def _legal_action_ids(frame: FrameData) -> list[int]:
        ids: list[int] = []
        for raw in frame.available_actions or []:
            try:
                value = int(raw.value if isinstance(raw, GameAction) else raw)
            except (AttributeError, TypeError, ValueError):
                continue
            if 1 <= value <= 7 and value not in ids:
                ids.append(value)
        if not ids:
            ids = [a.value for a in GameAction if a is not GameAction.RESET]
        return sorted(ids)

    @staticmethod
    def _aligned_delta(before: np.ndarray, after: np.ndarray) -> tuple[float, np.ndarray]:
        if before.shape == after.shape:
            mask = before != after
            return float(mask.mean()), mask
        height = max(before.shape[0], after.shape[0])
        width = max(before.shape[1], after.shape[1])
        old = np.full((height, width), 255, dtype=np.uint8)
        new = np.full((height, width), 254, dtype=np.uint8)
        old[: before.shape[0], : before.shape[1]] = before
        new[: after.shape[0], : after.shape[1]] = after
        mask = old != new
        return float(mask.mean()), mask

    def _reset_level_local(self, level: int) -> None:
        self.current_level = level
        self.level_start_step = self.step_index
        self.state_visits.clear()
        self.state_action_visits.clear()
        self.transitions.clear()
        self.state_q.clear()
        self.trace.clear()
        self.recent_change_mask = None
        self.macro_cursor = 0

    def _credit_progress(self) -> None:
        useful = [entry for entry in self.trace if entry.changed]
        suffix = useful[-20:]
        if suffix:
            self.success_macro = [
                (entry.candidate.action_id, entry.candidate.model_key)
                for entry in suffix
            ]
        for distance, entry in enumerate(reversed(list(self.trace)[-self.TRACE_CREDIT_DEPTH:])):
            credit = 12.0 * (0.86**distance)
            state_key = (entry.state_signature, entry.candidate.key)
            self.state_q[state_key] = min(20.0, self.state_q[state_key] + credit)
            self.role_q[entry.candidate.model_key] = min(
                12.0, self.role_q[entry.candidate.model_key] + 0.35 * credit
            )

    def _punish_failure_trace(self) -> None:
        for distance, entry in enumerate(reversed(list(self.trace)[-10:])):
            penalty = 4.0 * (0.72**distance)
            state_key = (entry.state_signature, entry.candidate.key)
            self.state_q[state_key] = max(-12.0, self.state_q[state_key] - penalty)
            self.role_q[entry.candidate.model_key] = max(
                -8.0, self.role_q[entry.candidate.model_key] - 0.2 * penalty
            )

    def _observe_pending(self, grid: np.ndarray, latest_frame: FrameData) -> None:
        if self.pending is None:
            return

        pending = self.pending
        candidate = pending.candidate
        delta, mask = self._aligned_delta(pending.grid, grid)
        changed = bool(delta > 0.0)
        levels = self._levels(latest_frame)
        progressed = bool(
            levels > pending.levels_completed or latest_frame.state is GameState.WIN
        )
        died = latest_frame.state is GameState.GAME_OVER
        new_signature = self._signature(grid)
        novel = self.state_visits[new_signature] == 0

        ev = self.evidence[candidate.key]
        ev.attempts += 1
        ev.changes += int(changed)
        ev.no_changes += int(not changed)
        ev.deaths += int(died)
        ev.progress += int(progressed)
        ev.delta_sum += delta
        ev.novelty_sum += float(novel)

        reward = -0.08
        reward += min(0.7, delta * 5.0)
        reward += 0.35 if changed and novel else 0.0
        reward += 25.0 if progressed else 0.0
        reward -= 8.0 if died else 0.0
        ev.value = 0.82 * ev.value + 0.18 * reward

        state_key = (pending.state_signature, candidate.key)
        old_q = self.state_q[state_key]
        self.state_q[state_key] = max(-15.0, min(25.0, old_q + 0.35 * (reward - old_q)))
        old_role = self.role_q[candidate.model_key]
        self.role_q[candidate.model_key] = max(
            -10.0, min(15.0, old_role + 0.12 * (reward - old_role))
        )
        self.transitions[state_key][new_signature] += 1
        self.trace.append(TraceEntry(pending.state_signature, candidate, changed, reward))
        self.recent_change_mask = mask
        self.last_effective_action_id = candidate.action_id if changed and not died else None
        if changed and not died and 1 <= candidate.action_id <= 4:
            self._learn_motion(pending.grid, grid)

        if progressed:
            self.progress_events += 1
            self._credit_progress()
        if died:
            self.death_events += 1
            self._punish_failure_trace()

        self.pending = None

    def _learn_motion(self, before: np.ndarray, after: np.ndarray) -> None:
        if before.shape != after.shape:
            return
        mask = before != after
        if not mask.any():
            return
        combined = np.concatenate((before[mask], after[mask]))
        values, counts = np.unique(combined, return_counts=True)
        terrain = int(values[int(np.argmax(counts))])
        new_object = mask & (after != terrain)
        if int(new_object.sum()) == 0:
            return
        ys, xs = np.nonzero(new_object)
        self.controlled_center = (
            int(round(float(xs.mean()))),
            int(round(float(ys.mean()))),
        )
        self.walkable_colors[terrain] += 1

    @staticmethod
    def _bucket(value: float, edges: tuple[float, ...]) -> int:
        return sum(value >= edge for edge in edges)

    def _components(self, grid: np.ndarray) -> list[dict[str, Any]]:
        height, width = grid.shape
        values, counts = np.unique(grid, return_counts=True)
        if len(values) == 0:
            return []
        background = int(values[int(np.argmax(counts))])
        frequency = {int(v): int(c) for v, c in zip(values, counts)}
        seen = np.zeros_like(grid, dtype=bool)
        components: list[dict[str, Any]] = []
        total = max(1, grid.size)

        for y0 in range(height):
            for x0 in range(width):
                color = int(grid[y0, x0])
                if color == background or seen[y0, x0]:
                    continue
                stack = [(y0, x0)]
                seen[y0, x0] = True
                cells: list[tuple[int, int]] = []
                while stack:
                    y, x = stack.pop()
                    cells.append((y, x))
                    for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                        if (
                            0 <= ny < height
                            and 0 <= nx < width
                            and not seen[ny, nx]
                            and int(grid[ny, nx]) == color
                        ):
                            seen[ny, nx] = True
                            stack.append((ny, nx))

                ys = [p[0] for p in cells]
                xs = [p[1] for p in cells]
                y1, y2 = min(ys), max(ys)
                x1, x2 = min(xs), max(xs)
                box_area = max(1, (y2 - y1 + 1) * (x2 - x1 + 1))
                area = len(cells)
                if area > int(0.55 * total):
                    continue
                density = area / box_area
                aspect = (x2 - x1 + 1) / max(1, y2 - y1 + 1)
                border = int(x1 == 0 or y1 == 0 or x2 == width - 1 or y2 == height - 1)
                color_rarity = 1.0 - frequency[color] / total
                smallness = 1.0 / math.sqrt(max(1, area))
                change_overlap = 0.0
                if self.recent_change_mask is not None and self.recent_change_mask.shape == grid.shape:
                    change_overlap = sum(bool(self.recent_change_mask[y, x]) for y, x in cells) / area
                mean_x = sum(xs) / area
                mean_y = sum(ys) / area
                center_y, center_x = min(
                    cells,
                    key=lambda point: (
                        (point[1] - mean_x) ** 2 + (point[0] - mean_y) ** 2,
                        point[0],
                        point[1],
                    ),
                )
                descriptor = (
                    "object",
                    color,
                    self._bucket(float(area), (2, 4, 9, 17, 33, 65, 129)),
                    self._bucket(aspect, (0.5, 0.8, 1.25, 2.0)),
                    self._bucket(density, (0.35, 0.7, 0.95)),
                    border,
                )
                score = 1.4 * color_rarity + 0.8 * smallness + 0.5 * density + 0.8 * change_overlap
                components.append(
                    {
                        "descriptor": descriptor,
                        "x": int(center_x),
                        "y": int(center_y),
                        "area": area,
                        "color": color,
                        "score": float(score),
                        "cells": cells,
                    }
                )

        components.sort(key=lambda item: (-item["score"], item["area"], item["y"], item["x"]))
        return components[: self.MAX_COMPONENTS]

    @staticmethod
    def _spatial_probes(grid: np.ndarray) -> list[tuple[int, int, str]]:
        height, width = grid.shape
        fractions = (0.125, 0.375, 0.625, 0.875)
        probes: list[tuple[int, int, str]] = []
        for row, fy in enumerate(fractions):
            for column, fx in enumerate(fractions):
                x = max(0, min(63, min(width - 1, int(round(fx * (width - 1))))))
                y = max(0, min(63, min(height - 1, int(round(fy * (height - 1))))))
                probes.append((x, y, f"region-{row}-{column}"))
        return probes

    def _navigation_preferences(self, grid: np.ndarray) -> dict[int, float]:
        if self.controlled_center is None or not self.walkable_colors:
            return {}
        terrain = self.walkable_colors.most_common(1)[0][0]
        traversable = grid == terrain
        height, width = grid.shape
        center_x, center_y = self.controlled_center

        starts: list[tuple[int, int, int]] = []
        for y in range(max(0, center_y - 6), min(height, center_y + 7)):
            for x in range(max(0, center_x - 6), min(width, center_x + 7)):
                if traversable[y, x]:
                    starts.append((abs(x - center_x) + abs(y - center_y), y, x))
        if not starts:
            return {}
        _, start_y, start_x = min(starts)
        start = (start_y, start_x)

        queue: deque[tuple[int, int]] = deque([start])
        distance: dict[tuple[int, int], int] = {start: 0}
        parent: dict[tuple[int, int], tuple[int, int]] = {}
        while queue:
            y, x = queue.popleft()
            for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                point = (ny, nx)
                if (
                    0 <= ny < height
                    and 0 <= nx < width
                    and traversable[ny, nx]
                    and point not in distance
                ):
                    distance[point] = distance[(y, x)] + 1
                    parent[point] = (y, x)
                    queue.append(point)

        targets: list[tuple[float, tuple[int, int]]] = []
        for component in self._components(grid):
            if component["color"] == terrain:
                continue
            if (component["x"] - center_x) ** 2 + (component["y"] - center_y) ** 2 <= 64:
                continue
            approaches: list[tuple[int, tuple[int, int]]] = []
            for y, x in component["cells"]:
                for point in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                    if point in distance:
                        approaches.append((distance[point], point))
            if not approaches:
                continue
            path_distance, approach = min(approaches)
            salience = float(component["score"])
            salience += 0.015 * min(80, int(component["area"]))
            salience -= 0.006 * path_distance
            targets.append((salience, approach))
        if not targets:
            return {}

        _, goal = max(targets, key=lambda item: (item[0], -distance[item[1]]))
        cursor = goal
        while parent.get(cursor) is not None and parent[cursor] != start:
            cursor = parent[cursor]
        dy = cursor[0] - start_y
        dx = cursor[1] - start_x
        if abs(dx) > abs(dy):
            action_id = 4 if dx > 0 else 3
        elif dy != 0:
            action_id = 2 if dy > 0 else 1
        elif dx != 0:
            action_id = 4 if dx > 0 else 3
        else:
            return {}
        return {action_id: 0.7}

    def _candidate_set(
        self, grid: np.ndarray, signature: str, latest_frame: FrameData
    ) -> list[Candidate]:
        legal = self._legal_action_ids(latest_frame)
        self.navigation_bonus = self._navigation_preferences(grid)
        candidates: list[Candidate] = []
        for action_id in legal:
            if action_id == 6:
                components = self._components(grid)
                occupied: set[tuple[int, int]] = set()
                for component in components:
                    descriptor = component["descriptor"]
                    x = max(0, min(63, int(component["x"])))
                    y = max(0, min(63, int(component["y"])))
                    occupied.add((x, y))
                    label = f"color={component['color']},area={component['area']}"
                    candidates.append(
                        Candidate(
                            action_id=6,
                            key=(6, descriptor, x, y),
                            model_key=("click", descriptor),
                            question="object-role",
                            target=(x, y),
                            target_label=label,
                            target_prior=float(component["score"]),
                        )
                    )
                for x, y, label in self._spatial_probes(grid):
                    if (x, y) in occupied:
                        continue
                    descriptor = ("region", label)
                    candidates.append(
                        Candidate(
                            action_id=6,
                            key=(6, descriptor, int(x), int(y)),
                            model_key=("click", descriptor),
                            question="spatial-intervention",
                            target=(int(x), int(y)),
                            target_label=label,
                            target_prior=0.08,
                        )
                    )
            else:
                question = "undo-causality" if action_id == 7 else "action-effect"
                candidates.append(
                    Candidate(
                        action_id=action_id,
                        key=(action_id,),
                        model_key=("simple", action_id),
                        question=question,
                    )
                )
        return candidates

    def _predicted_state(self, signature: str, key: tuple[Any, ...]) -> Optional[str]:
        counter = self.transitions.get((signature, key))
        if not counter:
            return None
        return counter.most_common(1)[0][0]

    def _score_candidates(self, candidates: list[Candidate], signature: str) -> None:
        for candidate in candidates:
            ev = self.evidence[candidate.key]
            exact_count = self.state_action_visits[(signature, candidate.key)]
            change_probability = (ev.changes + 1.0) / (ev.attempts + 2.0)
            progress_probability = (ev.progress + 0.05) / (ev.attempts + 2.0)
            death_probability = (ev.deaths + 0.15) / (ev.attempts + 2.0)
            information = 1.0 / math.sqrt(ev.attempts + exact_count + 1.0)
            predicted = self._predicted_state(signature, candidate.key)
            predicted_visits = self.state_visits[predicted] if predicted is not None else 0
            frontier = 1.25 if predicted is None else 1.0 / math.sqrt(predicted_visits + 1.0)
            no_effect_penalty = 0.0
            if ev.attempts >= 2:
                no_effect_penalty = 1.8 * (ev.no_changes / ev.attempts)
            repeat_penalty = 0.9 * math.log1p(exact_count)
            cycle_penalty = 0.55 * math.log1p(predicted_visits)
            undo_penalty = 0.65 if candidate.action_id == 7 and exact_count == 0 else 0.0

            macro_bonus = 0.0
            if self.macro_cursor < len(self.success_macro):
                wanted_action, wanted_model = self.success_macro[self.macro_cursor]
                if candidate.action_id == wanted_action and candidate.model_key == wanted_model:
                    macro_bonus = 2.2

            inertia_bonus = 0.0
            if candidate.action_id == self.last_effective_action_id and candidate.action_id <= 5:
                inertia_bonus = 0.45
            navigation = self.navigation_bonus.get(candidate.action_id, 0.0)
            if ev.attempts >= 2 and ev.no_changes / ev.attempts > 0.5:
                navigation = 0.0

            learned = 0.55 * self.state_q[(signature, candidate.key)]
            learned += 0.28 * self.role_q[candidate.model_key]
            utility = (
                learned
                + 3.0 * progress_probability
                + 0.75 * change_probability
                + 1.35 * information
                + frontier
                + 0.35 * candidate.target_prior
                + macro_bonus
                + inertia_bonus
                + navigation
                - 4.5 * death_probability
                - no_effect_penalty
                - repeat_penalty
                - cycle_penalty
                - undo_penalty
                - 0.12
            )
            candidate.utility = float(utility)
            candidate.information = float(information)
            candidate.risk = float(death_probability)
            candidate.confidence = float(
                max(0.0, min(1.0, 1.0 - information * 0.7 - death_probability * 0.3))
            )

    def _loaded_die(self, candidates: list[Candidate]) -> tuple[Candidate, str, float]:
        ordered = sorted(
            candidates,
            key=lambda c: (-c.utility, c.action_id, c.target or (-1, -1)),
        )
        best = ordered[0]
        gap = best.utility - ordered[1].utility if len(ordered) > 1 else math.inf
        best_ev = self.evidence[best.key]

        if best_ev.progress > 0 or (best_ev.attempts >= 3 and gap >= 1.25):
            print(f"[CAUSAL-DICE] die_roll=1.00000 mode=exploit selected={best.action_id} utility={best.utility:.5f}", flush=True)
            return best, "exploit", 1.0

        elapsed = max(0, self.step_index - self.level_start_step)
        temperature = max(0.18, 1.05 * math.exp(-elapsed / 140.0))
        selected = best
        selected_roll = -math.inf
        for candidate in ordered:
            uniform = min(1.0 - 1e-12, max(1e-12, self.rng.random()))
            gumbel = -math.log(-math.log(uniform))
            roll = candidate.utility + temperature * gumbel
            print(f"[CAUSAL-DICE] candidate action={candidate.action_id} utility={candidate.utility:.5f} gumbel={gumbel:.5f} roll={roll:.5f}", flush=True)
            if roll > selected_roll:
                selected = candidate
                selected_roll = roll
        print(f"[CAUSAL-DICE] die_roll={selected_roll:.5f} mode=explore selected={selected.action_id} utility={selected.utility:.5f}", flush=True)
        return selected, "explore", float(selected_roll)

    @staticmethod
    def _json_key(value: tuple[Any, ...]) -> str:
        return "/".join(str(part) for part in value)

    def _materialize_action(
        self,
        selected: Candidate,
        mode: str,
        die_roll: float,
        candidates: list[Candidate],
    ) -> GameAction:
        action = GameAction.from_id(int(selected.action_id))
        if selected.target is not None:
            x, y = selected.target
            action.set_data({"x": int(x), "y": int(y)})
        else:
            action.set_data({})

        alternatives = sorted(candidates, key=lambda c: -c.utility)[:6]
        action.reasoning = {
            "policy": "causal-dice-v1",
            "mode": mode,
            "question": selected.question,
            "selected": action.name,
            "target": list(selected.target) if selected.target is not None else None,
            "target_label": selected.target_label or None,
            "utility": round(selected.utility, 5),
            "information_gain": round(selected.information, 5),
            "estimated_risk": round(selected.risk, 5),
            "confidence": round(selected.confidence, 5),
            "die_roll": round(die_roll, 5),
            "alternatives": [
                {
                    "action": f"ACTION{c.action_id}",
                    "target": list(c.target) if c.target is not None else None,
                    "hypothesis": self._json_key(c.model_key),
                    "utility": round(c.utility, 5),
                }
                for c in alternatives
            ],
            "observed_progress_events": int(self.progress_events),
            "observed_failures": int(self.death_events),
        }
        return action

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        grid = self._grid_from(latest_frame)
        self._observe_pending(grid, latest_frame)
        levels = self._levels(latest_frame)

        if levels != self.current_level:
            self._reset_level_local(levels)

        print(
            f"[CAUSAL-DICE] game={self.game_id} level={levels} step={self.step_index + 1} "
            f"starting_evaluation state={latest_frame.state.name} candidates_pending={len(self.state_visits)}",
            flush=True,
        )

        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            if latest_frame.state is GameState.GAME_OVER:
                self.trace.clear()
                self.macro_cursor = 0
            reset = GameAction.RESET
            reset.set_data({})
            reset.reasoning = {
                "policy": "causal-dice-v1",
                "mode": "required-reset",
                "reason": latest_frame.state.value,
            }
            return reset

        signature = self._signature(grid)
        self.state_visits[signature] += 1
        candidates = self._candidate_set(grid, signature, latest_frame)
        if not candidates:
            fallback_id = self._legal_action_ids(latest_frame)[0]
            candidates = [
                Candidate(
                    action_id=fallback_id,
                    key=(fallback_id,),
                    model_key=("simple", fallback_id),
                    question="legal-fallback",
                )
            ]

        print(f"[CAUSAL-DICE] game={self.game_id} level={levels} step={self.step_index + 1} scoring {len(candidates)} candidates", flush=True)
        self._score_candidates(candidates, signature)
        selected, mode, die_roll = self._loaded_die(candidates)
        print(f"[CAUSAL-DICE] game={self.game_id} level={levels} step={self.step_index + 1} selected_action={selected.action_id} mode={mode} die_roll={die_roll:.5f}", flush=True)
        self.state_action_visits[(signature, selected.key)] += 1
        if self.macro_cursor < len(self.success_macro):
            wanted_action, wanted_model = self.success_macro[self.macro_cursor]
            if selected.action_id == wanted_action and selected.model_key == wanted_model:
                self.macro_cursor += 1

        action = self._materialize_action(selected, mode, die_roll, candidates)
        print(f"[CAUSAL-DICE] game={self.game_id} level={levels} step={self.step_index + 1} executed={action.name} target={selected.target} utility={selected.utility:.5f}", flush=True)
        dice_trace = [
            {
                "action": c.action_id,
                "target": list(c.target) if c.target is not None else None,
                "utility": round(c.utility, 5),
                "information": round(c.information, 5),
                "risk": round(c.risk, 5),
                "confidence": round(c.confidence, 5),
            }
            for c in sorted(candidates, key=lambda c: (-c.utility, c.action_id, c.target or (-1, -1)))
        ]
        print(
            f"[CAUSAL-DICE] game={self.game_id} level={levels} step={self.step_index + 1} "
            f"mode={mode} die_roll={die_roll:.5f} selected={selected.action_id} "
            f"target={selected.target} utility={selected.utility:.5f} candidates={dice_trace}",
            flush=True,
        )
        action.reasoning["dice_trace"] = dice_trace
        self.pending = Pending(signature, grid.copy(), levels, selected)
        self.step_index += 1
        return action


Writing /tmp/my_agent.py


In [3]:
import os
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logging.getLogger().setLevel(logging.INFO)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway sidecar to be ready.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework into a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent in as a framework template.
    !cp /tmp/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register MyAgent in the framework's agent registry. We rewrite
    # __init__.py because the upstream version eagerly imports
    # templates with deps we don't ship (langgraph, smolagents, etc.).
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Point the framework at the gateway sidecar.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    print("[KAGGLE] starting Causal Dice scored run with verbose logging", flush=True)
    # Run it. The gateway records every action and emits submission.parquet.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [4]:
import os
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logging.getLogger().setLevel(logging.INFO)

print('[KAGGLE] Causal Dice notebook loaded', flush=True)
print(f"[KAGGLE] competition_rerun={bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))}", flush=True)

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Validation mode: keep the run visible and emit an explicit diagnostic.
    candidate_envs = [
        Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files'),
        Path('/kaggle/input/environment_files'),
        Path('/data/data/com.termux/files/home/arc_agi3_data/environment_files'),
    ]
    env_root = next((p for p in candidate_envs if p.exists()), None)
    if env_root is None:
        print('[KAGGLE] no local environment_files found; writing dummy submission only', flush=True)
        import pandas as pd
        submission = pd.DataFrame(
            data=[['1_0', '1', True, 1]],
            columns=['row_id', 'game_id', 'end_of_game', 'score'])
        submission.to_parquet('/kaggle/working/submission.parquet', index=False)
        submission.head()
    else:
        print(f'[KAGGLE] validation dry-run using {env_root}', flush=True)
        print('[KAGGLE] this notebook does not execute the Kaggle gateway in validation mode', flush=True)
        print('[KAGGLE] repush/run as a competition rerun to see per-game planning and moves', flush=True)


[KAGGLE] Causal Dice notebook loaded
[KAGGLE] competition_rerun=False
[KAGGLE] validation dry-run using /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files
[KAGGLE] this notebook does not execute the Kaggle gateway in validation mode
[KAGGLE] repush/run as a competition rerun to see per-game planning and moves
